# randn-like-noise-source — faded example 3: Reparameterize with a dtype-matched, detached noise source

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `randn-like-noise-source`. Running the beacon reports progress on the `Generative: randn-like noise source` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: randn-like noise source` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`randn-like-noise-source`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "randn-like-noise-source"
DD_SUBTOPIC = "Generative: randn-like noise source"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A dtype 'leak' in reparameterization occurs when the noise `eps` is sampled with `torch.randn(*sigma.shape)` instead of `torch.randn_like(sigma)`. The output `z` inherits `float32` from `eps` even when `sigma` is `float64`, silently losing precision. An audit function checks whether `z.dtype` matches the input `sigma.dtype` for each test case.

## Faded exercise 3

### Exercise -- Reparameterization with a correct noise source

Implement `reparameterize(mu, sigma)` using the reparameterization trick:

`z = mu + sigma * eps`

The noise `eps` must come from `t.randn_like(sigma)` so it inherits `sigma`'s dtype and device (a bare `t.randn(...)` would silently fall back to float32). The noise is a constant sample, so detach it -- gradients should flow into `mu` and `sigma`, never into `eps`.

Fill in the body of `reparameterize`.

**Fill in:** Implement the reparameterization z = mu + sigma * eps, drawing eps from randn_like(sigma) so it matches sigma's dtype, and detaching eps so no gradient flows through the noise.

In [ ]:
import torch as t

def reparameterize(mu, sigma):
    raise NotImplementedError()  # TODO: return mu + sigma * eps, eps = detached randn_like(sigma)

t.manual_seed(35)
mu = t.zeros(4, 8, requires_grad=True)
sigma = t.ones(4, 8, requires_grad=True)
z = reparameterize(mu, sigma)
print('z dtype:', z.dtype, '| shape:', tuple(z.shape))


def _test():
    import torch as t

    # 1) dtype preserved across input dtypes (randn_like, not bare randn).
    for dtype in (t.float32, t.float64):
        mu = t.zeros(4, 8, dtype=dtype)
        sigma = t.ones(4, 8, dtype=dtype)
        z = reparameterize(mu, sigma)
        assert z.dtype == dtype, f'dtype leaked: expected {dtype}, got {z.dtype}'
        assert z.shape == sigma.shape

    # 2) Gradient flows into mu and sigma (noise must not block the path).
    mu = t.zeros(4, 8, requires_grad=True)
    sigma = t.ones(4, 8, requires_grad=True)
    z = reparameterize(mu, sigma)
    z.sum().backward()
    assert mu.grad is not None and sigma.grad is not None, 'grad must reach mu and sigma'
    assert t.allclose(mu.grad, t.ones_like(mu)), 'grad wrt mu must be ones'

    # 3) Stochastic: two calls give different samples (real noise drawn).
    t.manual_seed(0)
    a = reparameterize(t.zeros(100), t.ones(100))
    b = reparameterize(t.zeros(100), t.ones(100))
    assert not t.allclose(a, b), 'reparameterize must draw fresh noise'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def reparameterize(mu, sigma):
    # Noise matches sigma's dtype/device via randn_like, and is detached so
    # gradient flows into mu and sigma only (the reparameterization trick).
    eps = t.randn_like(sigma).detach()
    return mu + sigma * eps

t.manual_seed(35)
mu = t.zeros(4, 8, requires_grad=True)
sigma = t.ones(4, 8, requires_grad=True)
z = reparameterize(mu, sigma)
print('z dtype:', z.dtype, '| shape:', tuple(z.shape))
```
</details>